In [1]:
import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

train = pd.read_parquet(r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\parqueted_data\train_split')
val = pd.read_parquet(r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\parqueted_data\val_split')

In [2]:
# 컬럼 분리하기

# 시계열 컬럼군
time_cols = ['wl_t',          # 수위데이터 대표
        'sfc_rain_1h',   # 육상 누적강수데이터 대표
        'imerg_rain_1h', # 레이더 누적강수데이터 대표
        'radar_reflectivity_mean', # 레이더반사도 대표
        'radar_reflectivity_p95', # 레이더반사도 대표
        'target_maxrise_6h' # 종속변수: 수위상승량
        ]

# 수위 컬럼군
wl_cols = ['wl_t',
          'wl_t_minus_1h',
          'wl_t_minus_3h',
          'wl_t_minus_6h',
          'wl_t_minus_12h',
          'wl_t_minus_24h',
          ]

# 강수 컬럼군
rain_cols = [
            # IMERG강우
            'imerg_rain_1h',
            'imerg_rain_6h',
            'imerg_rain_24h',
            # 지상강우
            'sfc_rain_1h',
            'sfc_rain_6h',
            'sfc_rain_24h',
            # 레이더 반사도
            'radar_reflectivity_mean',
            'radar_reflectivity_p95',
            'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

In [ ]:
# ============================================================
# 1. train, val 데이터 박스플롯, 히스토그램 비교하기: 분포 비교용
# ============================================================

# 1-1. 시계열데이터부터 비교
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    len(time_cols),
    2,
    figsize = (12, 4 * len(time_cols))
)

if len(time_cols) == 1 :
    axes = [axes]

for i, col in enumerate(time_cols):
    tr = train[col].dropna()
    va = val[col].dropna()

    axes[i][0].boxplot([tr, va], tick_labels = ['Train', 'Validation'])
    axes[i][0].set_title(col, fontsize = 20)
    axes[i][0].tick_params(labelsize = 10)
    # 박스플롯은 x축에 이름이 적히므로 범례 필요없음

    axes[i][1].hist(tr, range = (min(tr.min(), va.min()), max(tr.max(), va.max())),
                    # range로 범위를 통일하지 않으면, 서로다른 범위를 bin만큼 쪼개므로 범위가 달라 분포도 바뀜
                    bins = 30, alpha = 0.5,  # 투명도(0이 완전 투명, 1이 완전불투명)
                    density = True,  # True는 두 데이터의 n이 다를때 전체 면적은 동일하게 만들어 비교케 해줌
                    label = 'Train')
    axes[i][1].hist(va, range = (min(tr.min(), va.min()), max(tr.max(), va.max())),
                    bins = 30, alpha = 0.5,  # 투명도(0이 완전 투명, 1이 완전불투명)
                    density = True,  # True는 두 데이터의 n이 다를때 전체 면적은 동일하게 만들어 비교케 해줌
                    label = 'Validation')
    axes[i][1].set_title(col, fontsize = 20)
    axes[i][1].tick_params(labelsize = 10)
    axes[i][1].legend()  # 범례

plt.tight_layout()
plt.show()

In [23]:
# ============================================================
# 2. 기초통계량 비교
# ============================================================

for col in (time_cols + ['imerg_rain_6h', 'imerg_rain_24h']):
    print(col)
    print(pd.concat(
        [
        train[col].quantile([0.5, 0.75, 0.9, 0.95, 0.99, 1.0]),
        val[col].quantile([0.5, 0.75, 0.9, 0.95, 0.99, 1.0])
        ],
        axis = 1,  # 가로로 붙이기
        keys = ['Train', 'Validation']  # 시리즈는 columns가 아니라 keys로 명명
    ))

    print()

wl_t
         Train  Validation
0.50    1.1400        1.11
0.75    1.9800        1.90
0.90    4.0100        3.72
0.95   25.0100       24.96
0.99   78.8069       79.28
1.00  139.6400      136.81

sfc_rain_1h
      Train  Validation
0.50    0.0         0.0
0.75    1.0         0.6
0.90    3.5         3.5
0.95    6.0         6.5
0.99   15.5        16.0
1.00   86.2        87.0

imerg_rain_1h
        Train  Validation
0.50    0.250     0.14200
0.75    1.517     1.34800
0.90    4.133     4.08900
0.95    6.524     6.84540
0.99   13.370    14.56216
1.00  114.530    49.91000

radar_reflectivity_mean
       Train  Validation
0.50  13.253    11.93800
0.75  22.814    21.85500
0.90  30.009    28.91200
0.95  33.465    32.77800
0.99  38.757    38.89352
1.00  55.917    52.05500

radar_reflectivity_p95
        Train  Validation
0.50  21.9900      21.880
0.75  29.4400      28.760
0.90  35.9109      35.588
0.95  39.3300      39.510
0.99  46.1700      46.530
1.00  67.7700      58.320

target_maxrise_6h
   

In [21]:
print(
    train.groupby('event_id')['imerg_rain_24h'].max().describe()
)

print(
    val.groupby('event_id')['imerg_rain_24h'].max().describe()
)

count     98.000000
mean      65.290745
std       66.528920
min        4.455000
25%       27.502250
50%       43.565000
75%       79.712750
max      371.228000
Name: imerg_rain_24h, dtype: float64
count     24.000000
mean      69.391625
std       47.694795
min        8.520000
25%       34.315250
50%       63.704000
75%      109.294250
max      161.745000
Name: imerg_rain_24h, dtype: float64


In [22]:
print(
    train.groupby('event_id')['imerg_rain_24h'].max()
         .quantile([0.5, 0.75, 0.9, 0.95, 1.0])
)

print(
    val.groupby('event_id')['imerg_rain_24h'].max()
       .quantile([0.5, 0.75, 0.9, 0.95, 1.0])
)

0.50     43.56500
0.75     79.71275
0.90    129.21310
0.95    172.88000
1.00    371.22800
Name: imerg_rain_24h, dtype: float64
0.50     63.70400
0.75    109.29425
0.90    141.23240
0.95    154.02405
1.00    161.74500
Name: imerg_rain_24h, dtype: float64
